<table style="width: 100%; border-collapse: collapse; border: none; background: #f8fafc; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #0f172a; font-size: 2.1em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        Taller Práctico: Auditoría de Datos y Metodología CRISP-DM (Hands-On) 📝🔍
      </h1>
      <p style="margin: 6px 0 0 0; color: #f59e0b; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Taller Práctico Evaluativo 📝 | Taller Evaluativo
      </p>
      <p style="margin: 4px 0 0 0; color: #64748b; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #d97706; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        Módulo 00 • Data Mining
      </span><br>
      <span style="color: #64748b; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #f59e0b; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Data%20Mining/00%20-%20Introduccion%20a%20la%20Mineria%20de%20Datos/homeworks/00_KDD_CRISP_DM_Hands_On.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## 🎯 Objetivos del Taller

En este taller asumirás el rol de **Lead Data Scientist** a cargo del diagnóstico inicial de un proyecto de minería de datos empresarial:
1. **Mapeo Metodológico:** Diagnóstico de objetivos de negocio y traducción a las 6 fases de CRISP-DM.
2. **Auditoría de Calidad:** Diagnóstico exhaustivo de los 5 pilares (Completitud, Conformidad, Consistencia, Unicidad).
3. **Saneamiento de Datos:** Pipeline de limpieza y resolución de anomalías detectadas.
4. **Taxonomía de Tareas:** Asignación fundamentada de algoritmos según el tipo de problema.
5. **Gobernanza y Ética:** Implementación de un pipeline de seudonimización criptográfica para datos sensibles.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os, warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (8.5, 4.5)

def load_dataset(filename):
    for p in [os.path.join("data", filename), os.path.join("..", "data", filename)]:
        if os.path.exists(p): return p
    return os.path.join("data", filename)

print("🚀 Entorno de Taller KDD & CRISP-DM inicializado con éxito.")

---
### 📌 Parte 1: Auditoría de Calidad del Dato (`auditoria_calidad_datos.csv`)

**Ejercicio 1.1:** Carga el archivo `auditoria_calidad_datos.csv`. Calcula el porcentaje exacto de nulos por columna, identifica cuántas filas están duplicadas y detecta cuántos clientes tienen edades imposibles (< 0 o > 120 años).

In [ ]:
### TU CÓDIGO AQUÍ ###
df_raw = pd.read_csv(load_dataset('auditoria_calidad_datos.csv'))
print("Forma original:", df_raw.shape)

pct_nulos = df_raw.isnull().mean() * 100
print("\nPorcentaje de Nulos:\n", pct_nulos.round(2))

duplicados = df_raw.duplicated().sum()
print(f"\nFilas duplicadas exactas: {duplicados}")

edades_invalidas = df_raw[(df_raw['edad'] < 0) | (df_raw['edad'] > 120)].shape[0]
print(f"Registros con edad inválida: {edades_invalidas}")

---
### 📌 Parte 2: Pipeline de Limpieza y Estandarización

**Ejercicio 2.1:** Diseña una función de saneamiento que:
1. Elimine las filas duplicadas.
2. Reemplace las edades imposibles por `np.nan` y luego las impute con la mediana de las edades válidas.
3. Homogenice la columna `ciudad` a mayúsculas y limpie inconsistencias.

In [ ]:
### TU CÓDIGO AQUÍ ###
df_clean = df_raw.drop_duplicates().copy()

# Corregir edades
df_clean.loc[(df_clean['edad'] < 0) | (df_clean['edad'] > 120), 'edad'] = np.nan
df_clean['edad'] = df_clean['edad'].fillna(df_clean['edad'].median())

# Homogeneizar ciudades
df_clean['ciudad'] = df_clean['ciudad'].astype(str).str.upper().str.strip()
df_clean['ciudad'] = df_clean['ciudad'].replace({'DESCONOCIDO': np.nan, 'NONE': np.nan})

print("Forma tras limpieza:", df_clean.shape)
display(df_clean.head(5))

---
### 📌 Parte 3: Seudonimización Ética de Identificadores (Habeas Data)

**Ejercicio 3.1:** Aplica hashing criptográfico SHA-256 con sal para transformar la columna `id_cliente` en un identificador pseudo-anónimo irrevocable.

In [ ]:
import hashlib

### TU CÓDIGO AQUÍ ###
def anonimizar(val, salt="USTA_TUNJA_2026"):
    return hashlib.sha256(f"{val}_{salt}".encode('utf-8')).hexdigest()[:12]

df_clean['id_anonimo'] = df_clean['id_cliente'].apply(anonimizar)
display(df_clean[['id_cliente', 'id_anonimo', 'edad', 'ciudad']].head(5))

---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Minería de Datos (Data Mining)</i>
  </p>
</div>